> **Note**: This notebook trains YOLOv11 models for general Waste Detection. It supports model comparison (yolo11s vs yolo11m), experiment tracking, ONNX export, and automatic checkpoint resumption.

# 🏋️ Waste Detection — YOLOv11 Training (Notebook 06)

### Overview
Train YOLOv11 on the augmented waste detection dataset. Compare model variants, track experiments, and export the best model.

### Pipeline Position
```
NB 05 (Augmentation) → [taco_yolo_augmented/] → NB 06 (THIS) → [models/best.pt] → NB 07, 08, 09
```

### Training Strategy
1. Train YOLOv11s as baseline
2. Optionally train YOLOv11m for comparison
3. Compare metrics and select the best model
4. Export best model to ONNX format

## 1. Environment Setup

In [ ]:
!pip install -q ultralytics onnx onnxruntime onnxslim rich tqdm pyyaml pandas opencv-python matplotlib

In [ ]:
import os
import sys
import json
import time
import shutil
import datetime
from pathlib import Path

import yaml
import torch
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from ultralytics import YOLO

console = Console()

## 2. Hardware Detection

In [ ]:
# Detect best available device
if torch.cuda.is_available():
    DEVICE = 'cuda:0'
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    console.print(f"[green]✔ CUDA GPU detected: {GPU_NAME} ({GPU_MEM:.1f} GB)[/green]")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
    console.print("[green]✔ Apple Silicon MPS detected[/green]")
else:
    DEVICE = 'cpu'
    console.print("[yellow]⚠ No GPU detected — training will be slow[/yellow]")

console.print(f"[cyan]Device: {DEVICE}[/cyan]")
console.print(f"[cyan]PyTorch: {torch.__version__}[/cyan]")

## 3. Dataset & Path Configuration

In [ ]:
# ==========================================
# Paths
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')
DATASET_DIR = PROJECT_ROOT / 'datasets/taco_yolo_augmented'
DATASET_YAML = DATASET_DIR / 'data.yaml'
MODELS_DIR = PROJECT_ROOT / 'models'
TRAIN_LOGS_DIR = PROJECT_ROOT / 'results/training'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_LOGS_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================
# Verify dataset
# ==========================================
if not DATASET_YAML.exists():
    raise FileNotFoundError(f"data.yaml not found at {DATASET_YAML}. Run Notebook 05 first.")

with open(DATASET_YAML, 'r') as f:
    dataset_config = yaml.safe_load(f)

NUM_CLASSES = len(dataset_config.get('names', {}))
console.print(f"[green]✔ Dataset verified: {NUM_CLASSES} classes[/green]")
console.print(f"[cyan]  YAML: {DATASET_YAML}[/cyan]")

## 4. Training Configuration
Hyperparameters are set conservatively. Adjust based on your GPU capabilities.

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Epochs | 120 | Sufficient for convergence with early stopping |
| Image Size | 832 | Better small-object detection than 640 |
| Patience | 25 | Stop early if no improvement |
| Batch Size | 8 | Safe for T4/MPS; increase if GPU allows |

In [ ]:
# ==========================================
# Training Hyperparameters
# ==========================================
EPOCHS = 120
BATCH_SIZE = 8
IMAGE_SIZE = 832
PATIENCE = 25
OPTIMIZER = 'AdamW'
SEED = 42

# MPS-specific settings
MIXED_PRECISION = DEVICE == 'cuda:0'  # AMP only reliable on CUDA
CACHE_IMAGES = True
WORKERS = 0 if DEVICE == 'mps' else 4  # macOS multiprocessing issues

console.print(Panel.fit(
    f"[bold cyan]Training Configuration[/bold cyan]\n"
    f"Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | ImgSz: {IMAGE_SIZE}\n"
    f"Optimizer: {OPTIMIZER} | Patience: {PATIENCE} | Seed: {SEED}\n"
    f"Device: {DEVICE} | AMP: {MIXED_PRECISION} | Cache: {CACHE_IMAGES}"
))

## 5. Model Initialization & Training

In [ ]:
def train_model(model_variant: str, experiment_name: str) -> dict:
    """Train a YOLO model and return results summary."""
    console.print(f"\n[bold cyan]{'='*50}[/bold cyan]")
    console.print(f"[bold cyan]  Training: {model_variant} → {experiment_name}[/bold cyan]")
    console.print(f"[bold cyan]{'='*50}[/bold cyan]")

    # Check for resume checkpoint
    project_dir = TRAIN_LOGS_DIR / experiment_name
    last_checkpoint = project_dir / 'weights' / 'last.pt'

    if last_checkpoint.exists():
        console.print(f"[yellow]⚠ Resuming from {last_checkpoint}[/yellow]")
        model = YOLO(str(last_checkpoint))
        resume = True
    else:
        console.print(f"[green]✔ Starting fresh: {model_variant}[/green]")
        model = YOLO(model_variant)
        resume = False

    model.info()

    start_time = time.time()
    start_fmt = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    console.print(f"[green]Training started at {start_fmt}[/green]")

    try:
        results = model.train(
            data=str(DATASET_YAML),
            epochs=EPOCHS,
            batch=BATCH_SIZE,
            imgsz=IMAGE_SIZE,
            patience=PATIENCE,
            optimizer=OPTIMIZER,
            seed=SEED,
            device=DEVICE,
            amp=MIXED_PRECISION,
            cache=CACHE_IMAGES,
            workers=WORKERS,
            project=str(TRAIN_LOGS_DIR),
            name=experiment_name,
            resume=resume,
            exist_ok=True,
            plots=True,
            save=True,
            val=True
        )
    except torch.cuda.OutOfMemoryError:
        console.print("[bold red]OOM ERROR: Reduce BATCH_SIZE or IMAGE_SIZE[/bold red]")
        raise
    except KeyboardInterrupt:
        console.print("[yellow]Training interrupted by user[/yellow]")
        results = None

    duration = time.time() - start_time
    hours, remainder = divmod(duration, 3600)
    minutes, seconds = divmod(remainder, 60)

    console.print(f"[green]Training completed in {int(hours)}h {int(minutes)}m {int(seconds)}s[/green]")

    # Extract best metrics
    report = {
        'model_variant': model_variant,
        'experiment_name': experiment_name,
        'training_duration_s': round(duration, 2),
        'epochs_configured': EPOCHS,
        'image_size': IMAGE_SIZE,
        'batch_size': BATCH_SIZE,
        'device': DEVICE,
    }

    results_csv = TRAIN_LOGS_DIR / experiment_name / 'results.csv'
    best_pt = TRAIN_LOGS_DIR / experiment_name / 'weights' / 'best.pt'

    if results_csv.exists():
        df = pd.read_csv(results_csv)
        df.columns = df.columns.str.strip()
        try:
            best_idx = df['metrics/mAP50-95(B)'].idxmax()
            best = df.iloc[best_idx]
            report.update({
                'total_epochs': len(df),
                'best_epoch': int(best.get('epoch', best_idx)),
                'precision': round(float(best.get('metrics/precision(B)', 0)), 4),
                'recall': round(float(best.get('metrics/recall(B)', 0)), 4),
                'mAP50': round(float(best.get('metrics/mAP50(B)', 0)), 4),
                'mAP50_95': round(float(best.get('metrics/mAP50-95(B)', 0)), 4),
            })
        except Exception as e:
            console.print(f"[yellow]Could not parse results: {e}[/yellow]")

    if best_pt.exists():
        report['best_pt_path'] = str(best_pt)
        report['model_size_mb'] = round(best_pt.stat().st_size / (1024*1024), 2)

    return report

In [ ]:
# ==========================================
# Experiment 1: YOLOv11s (baseline)
# ==========================================
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
exp1_name = f"yolo11s_waste_{timestamp}"

report_s = train_model('yolo11s.pt', exp1_name)

# Save experiment report
with open(MODELS_DIR / f'{exp1_name}_report.json', 'w') as f:
    json.dump(report_s, f, indent=4)

console.print(Panel.fit(
    f"[bold green]YOLOv11s Training Complete[/bold green]\n\n"
    f"mAP50: {report_s.get('mAP50', 'N/A')}\n"
    f"mAP50-95: {report_s.get('mAP50_95', 'N/A')}\n"
    f"Precision: {report_s.get('precision', 'N/A')}\n"
    f"Recall: {report_s.get('recall', 'N/A')}"
))

## 6. Optional: Train YOLOv11m for Comparison
Uncomment and run the cell below to train YOLOv11m. This takes significantly longer but may yield better accuracy.

> **Note**: Only run this if you have sufficient GPU time and memory. YOLOv11m is ~3x larger than YOLOv11s.

In [ ]:
# ==========================================
# Experiment 2: YOLOv11m (optional — uncomment to run)
# ==========================================
# exp2_name = f"yolo11m_waste_{timestamp}"
# report_m = train_model('yolo11m.pt', exp2_name)
#
# with open(MODELS_DIR / f'{exp2_name}_report.json', 'w') as f:
#     json.dump(report_m, f, indent=4)
#
# console.print(Panel.fit(
#     f"[bold green]YOLOv11m Training Complete[/bold green]\n\n"
#     f"mAP50: {report_m.get('mAP50', 'N/A')}\n"
#     f"mAP50-95: {report_m.get('mAP50_95', 'N/A')}"
# ))

## 7. Export Best Model
Copy `best.pt` to the central `models/` directory and export to ONNX format.

In [ ]:
# ==========================================
# Copy best.pt to models/ directory
# ==========================================
best_pt_source = TRAIN_LOGS_DIR / exp1_name / 'weights' / 'best.pt'

if best_pt_source.exists():
    shutil.copy2(str(best_pt_source), str(MODELS_DIR / 'best.pt'))
    console.print(f"[green]✔ best.pt copied to {MODELS_DIR / 'best.pt'}[/green]")

    # Also copy last.pt
    last_pt_source = TRAIN_LOGS_DIR / exp1_name / 'weights' / 'last.pt'
    if last_pt_source.exists():
        shutil.copy2(str(last_pt_source), str(MODELS_DIR / 'last.pt'))

    # Save training config
    training_config = {
        'model_variant': report_s['model_variant'],
        'experiment_name': exp1_name,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'image_size': IMAGE_SIZE,
        'optimizer': OPTIMIZER,
        'device': DEVICE,
        'dataset_path': str(DATASET_DIR),
        'num_classes': NUM_CLASSES,
        'training_duration_s': report_s.get('training_duration_s', 0),
    }

    with open(MODELS_DIR / 'training_configuration.json', 'w') as f:
        json.dump(training_config, f, indent=4)

    console.print("[green]✔ Training configuration saved[/green]")
else:
    console.print("[bold red]✖ best.pt not found — training may have failed[/bold red]")

## 8. ONNX Export
Export the best model to ONNX format for deployment.

In [ ]:
# ==========================================
# ONNX Export
# ==========================================
best_pt_path = MODELS_DIR / 'best.pt'

if best_pt_path.exists():
    try:
        model_export = YOLO(str(best_pt_path))
        onnx_path = model_export.export(format='onnx', imgsz=IMAGE_SIZE, simplify=True)
        console.print(f"[green]✔ ONNX model exported: {onnx_path}[/green]")

        # Copy to models directory
        if onnx_path and Path(onnx_path).exists():
            shutil.copy2(onnx_path, str(MODELS_DIR / 'best.onnx'))
            onnx_size = Path(onnx_path).stat().st_size / (1024*1024)
            console.print(f"[green]✔ best.onnx saved ({onnx_size:.1f} MB)[/green]")

            # Verify ONNX predictions
            console.print("[cyan]Verifying ONNX model...[/cyan]")
            onnx_model = YOLO(str(MODELS_DIR / 'best.onnx'))
            test_img_dir = DATASET_DIR / 'images' / 'test'
            test_imgs = list(test_img_dir.glob('*.[jJ][pP][gG]'))[:1]
            if test_imgs:
                onnx_results = onnx_model.predict(source=str(test_imgs[0]), conf=0.25, verbose=False)
                num_dets = len(onnx_results[0].boxes) if onnx_results[0].boxes is not None else 0
                console.print(f"[green]✔ ONNX verification: {num_dets} detections on test image[/green]")
    except Exception as e:
        console.print(f"[yellow]⚠ ONNX export failed: {e}[/yellow]")
        console.print("[yellow]  This is non-critical — best.pt is still available.[/yellow]")
else:
    console.print("[red]✖ Cannot export — best.pt not found[/red]")

## 9. Training Curves & Visualization

In [ ]:
# ==========================================
# Display training curves
# ==========================================
def display_training_metrics(metrics_dir: Path):
    plots = {
        'Training Curves': metrics_dir / 'results.png',
        'Confusion Matrix': metrics_dir / 'confusion_matrix.png',
        'F1 Curve': metrics_dir / 'F1_curve.png',
        'Precision-Recall Curve': metrics_dir / 'PR_curve.png'
    }

    for title, plot_path in plots.items():
        if plot_path.exists():
            try:
                img = cv2.imread(str(plot_path))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                plt.figure(figsize=(15, 10))
                plt.imshow(img)
                plt.title(title, fontweight='bold', fontsize=16)
                plt.axis('off')
                plt.tight_layout()
                plt.show()
            except Exception as e:
                console.print(f"[yellow]Could not render {title}: {e}[/yellow]")
        else:
            console.print(f"[yellow]Plot not found: {title}[/yellow]")

exp_dir = TRAIN_LOGS_DIR / exp1_name
display_training_metrics(exp_dir)

## 10. Final Training Summary

In [ ]:
# ==========================================
# Final summary
# ==========================================
console.print(Panel.fit(
    f"[bold green]Model Training Complete[/bold green]\n\n"
    f"[bold cyan]✔ Model:[/bold cyan] {report_s['model_variant']}\n"
    f"[bold cyan]✔ Epochs:[/bold cyan] {report_s.get('total_epochs', 'N/A')}\n"
    f"[bold cyan]✔ Best Epoch:[/bold cyan] {report_s.get('best_epoch', 'N/A')}\n"
    f"[bold magenta]✔ Precision:[/bold magenta] {report_s.get('precision', 'N/A')}\n"
    f"[bold magenta]✔ Recall:[/bold magenta] {report_s.get('recall', 'N/A')}\n"
    f"[bold yellow]✔ mAP@50:[/bold yellow] {report_s.get('mAP50', 'N/A')}\n"
    f"[bold yellow]✔ mAP@50-95:[/bold yellow] {report_s.get('mAP50_95', 'N/A')}\n\n"
    f"[bold]Model Size:[/bold] {report_s.get('model_size_mb', 'N/A')} MB\n"
    f"[bold]Saved:[/bold] {MODELS_DIR / 'best.pt'}\n"
    f"[bold]ONNX:[/bold] {MODELS_DIR / 'best.onnx'}\n\n"
    f"[bold magenta]Next Notebook:[/bold magenta] 07_Model_Evaluation.ipynb"
))

## 🍎 Alternative: Training on Apple Silicon (M1/M2/M3/M4)
If running locally on a MacBook, use `device='mps'`. The configuration above auto-detects MPS.

In [ ]:
# Apple Silicon is auto-detected in the hardware section above.
# Key adjustments for MPS:
#   - WORKERS = 0 (prevents dataloader freezing)
#   - MIXED_PRECISION = False (AMP unreliable on MPS)
#   - CACHE_IMAGES = True (fast unified memory)
if DEVICE == 'mps':
    console.print("[cyan]ℹ Running on Apple Silicon — settings auto-adjusted[/cyan]")